In [3]:
import json
import os
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [4]:
os.listdir('/mnt/')

['Gbenga_Enemy',
 'Context_testing',
 'mlp_valid_ll3_90b.json',
 'llava7b_emb_3ctx.csv',
 'Manifold',
 'grounded_sam_data',
 'maskrcnn_models',
 'contexttesting.tar.gz',
 'mlp_test_ll3_90b.json',
 'hateful_memes',
 'valid_llama3_90b.json',
 'Software',
 'reformat_json.ipynb',
 'test_llama3_90b.json',
 '.ipynb_checkpoints',
 'point2mesh']

In [5]:
data_path = '/mnt'
val_set = os.path.join(data_path, 'mlp_valid_ll3_90b.json')
test_set = os.path.join(data_path, 'mlp_test_ll3_90b.json')
# Load and preprocess the test data
with open(val_set, 'r') as file:
    val_data = json.load(file)

# Prepare the feature vectors and labels for the test set
val_feature_vectors = []
val_labels = []

for image_id, info in val_data.items():
    val_feature_vectors.append(info['answers_vector'])
    val_labels.append(1 if info['label'] == "4" else 0)

In [10]:
import torch.nn as nn
import torch.optim as optim

# Convert to torch tensors
X = torch.tensor(np.array(val_feature_vectors), dtype=torch.float32)
y = torch.tensor(val_labels, dtype=torch.long)

# Define the MLP model
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 2)  # Output 2 classes

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Instantiate the model
input_size = X.shape[1]
model = MLP(input_size)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 100
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 0.6752
Epoch [20/100], Loss: 0.6564
Epoch [30/100], Loss: 0.6368
Epoch [40/100], Loss: 0.6120
Epoch [50/100], Loss: 0.5874
Epoch [60/100], Loss: 0.5700
Epoch [70/100], Loss: 0.5604
Epoch [80/100], Loss: 0.5556
Epoch [90/100], Loss: 0.5522
Epoch [100/100], Loss: 0.5489


In [11]:
# Load and preprocess the test data
with open(test_set, 'r') as file:
    test_data = json.load(file)

# Prepare the feature vectors and labels for the test set
test_feature_vectors = []
test_labels = []

for image_id, info in test_data.items():
    test_feature_vectors.append(info['answers_vector'])
    test_labels.append(1 if info['label'] == "4" else 0)

# Convert to torch tensors
X_test = torch.tensor(np.array(test_feature_vectors), dtype=torch.float32)
y_test = torch.tensor(test_labels, dtype=torch.long)

# Set model to evaluation mode
model.eval()
with torch.no_grad():
    # Predict on the test data
    test_outputs = model(X_test)
    _, predicted = torch.max(test_outputs, 1)  # Get the predicted labels

# Compute evaluation metrics
accuracy = accuracy_score(y_test, predicted)
precision = precision_score(y_test, predicted)
recall = recall_score(y_test, predicted)
f1 = f1_score(y_test, predicted)
macro_f1 = f1_score(y_test, predicted, average='macro')

# Confusion matrix for TP, FP, TN, FN
tn, fp, fn, tp = confusion_matrix(y_test, predicted).ravel()

# Print all metrics
print(f"Test Accuracy: {accuracy * 100:.3f}%")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")
print(f"Macro F1 Score: {macro_f1:.3f}")
print(f"True Positives: {tp:.3f}")
print(f"False Positives: {fp:.3f}")
print(f"True Negatives: {tn:.3f}")
print(f"False Negatives: {fn:.3f}")

Test Accuracy: 64.63%
Precision: 0.70
Recall: 0.66
F1 Score: 0.68
Macro F1 Score: 0.64
True Positives: 117
False Positives: 50
True Negatives: 84
False Negatives: 60


In [14]:
# Print all metrics
print(f"MLP Results on Testing Set")
print(f"Test Accuracy: {accuracy * 100:.3f}%")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")
print(f"Macro F1 Score: {macro_f1:.3f}")
print(f"True Positives: {tp:.3f}")
print(f"False Positives: {fp:.3f}")
print(f"True Negatives: {tn:.3f}")
print(f"False Negatives: {fn:.3f}")

MLP Results on Testing Set
Test Accuracy: 64.630%
Precision: 0.701
Recall: 0.661
F1 Score: 0.680
Macro F1 Score: 0.642
True Positives: 117.000
False Positives: 50.000
True Negatives: 84.000
False Negatives: 60.000
